# Reproduce the article's Soubré + Sefwi-Wiawso numbers (snapshot 2026-06-08)

This notebook loads the **2026-06-08 snapshot** and reproduces the quantitative claims of both companion articles:

- [`eudr-pixel-auditierbarkeit`](https://mybytes.com/research/notes/eudr-pixel-auditierbarkeit) — Plot 2 region comparison (Hansen 2025 vintage).
- [`eudr-update-2026`](https://mybytes.com/research/notes/eudr-update-2026) — Plot 2 vintage-drift (Hansen 2024 vs 2025).

Every number you see here comes from `data/runs/2026-06-08/area_summary.csv` and `vintage_drift.csv`, which together are the **source of truth** for every quantitative claim in the articles.

If any assert in section 3 fails, the article and the snapshot have drifted apart — that is a publishability gate.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SNAPSHOT_DIR = REPO_ROOT / 'data' / 'runs' / '2026-06-08'
print(f'Snapshot: {SNAPSHOT_DIR}')

## 1 · Load the published snapshot

In [ ]:
area = pd.read_csv(SNAPSHOT_DIR / 'area_summary.csv')
drift = pd.read_csv(SNAPSHOT_DIR / 'vintage_drift.csv')
audit = pd.read_csv(SNAPSHOT_DIR / 'audit_trail_sample.csv')
area

In [ ]:
drift

## 2 · Audit-trail sample (Plot 3 of `eudr-pixel-auditierbarkeit`)

In [ ]:
audit

## 3 · Publishability gate — asserts on every article number

The block below asserts that the snapshot reproduces:

- The Hansen-2025 vintage risk shares quoted in `eudr-pixel-auditierbarkeit` (Soubré **2.74 %**, Sefwi-Wiawso **6.05 %**).
- The vintage-drift quoted in `eudr-update-2026` (Soubré **2.28 → 2.74 %**, Sefwi-Wiawso **5.13 → 6.05 %**).

Tolerance is **± 0.05 percentage points** to absorb FDP-rounding noise.

In [ ]:
EXPECTED_2025 = {
    'civ_soubre_33km': 2.74,
    'gha_sefwi_wiawso_33km': 6.05,
}
EXPECTED_DRIFT = {
    'civ_soubre_33km': (2.28, 2.74),
    'gha_sefwi_wiawso_33km': (5.13, 6.05),
}
TOL = 0.05

vintage_2025 = area[area['hansen_vintage'] == 'hansen_v2025_v1_13'].set_index('aoi_id')
for aoi_id, expected in EXPECTED_2025.items():
    actual = float(vintage_2025.loc[aoi_id, 'risk_share_pct'])
    assert abs(actual - expected) < TOL, (
        f'2025 vintage mismatch for {aoi_id}: snapshot {actual:.2f}% vs article {expected:.2f}%'
    )
    print(f'{aoi_id} 2025-vintage: {actual:.2f} %  ✔  matches article')

drift_idx = drift.set_index('aoi_id')
for aoi_id, (exp_24, exp_25) in EXPECTED_DRIFT.items():
    act_24 = float(drift_idx.loc[aoi_id, 'risk_share_2024_pct'])
    act_25 = float(drift_idx.loc[aoi_id, 'risk_share_2025_pct'])
    assert abs(act_24 - exp_24) < TOL, f'2024 vintage mismatch {aoi_id}: {act_24} vs {exp_24}'
    assert abs(act_25 - exp_25) < TOL, f'2025 vintage mismatch {aoi_id}: {act_25} vs {exp_25}'
    print(f'{aoi_id} drift: {act_24:.2f} → {act_25:.2f} %  ✔  matches article')

print('\nAll article numbers reproduce from the 2026-06-08 snapshot. Publishability gate green.')